In [1]:
from nero_interface_server import NeroDualArmServer

In [2]:
import logging
log = logging.getLogger(__name__)

In [3]:
import numpy as np
import time

In [4]:
server = NeroDualArmServer(gripper_enabled=True)

[SERVER] Failed to connect to right arm: CAN socket can_right does not exist.


[Left Arm] 正在获取当前关节角作为 IK 初始基准...
[Left Arm] IK Solver 初始化完成！初始关节角: [-0.437  1.408 -0.343  0.911 -0.189 -0.557 -1.081]


In [5]:
fp = server.left_robot.get_flange_pose()
robot_pose = np.array(fp.msg)

print("robot z:", robot_pose[2])

robot z: 0.015592


In [6]:
assert server.left_robot is not None, "Left arm failed"
assert server.right_robot is not None, "Right arm failed"
print("Left robot:", server.left_robot)
print("Right robot:", server.right_robot)

Left robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f01e5c7ad50>
Right robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f01e63f7cd0>


In [7]:
left_joints = server.left_robot_get_joint_positions()
right_joints = server.right_robot_get_joint_positions()
print("Left joints:", left_joints)
print("Right joints:", right_joints)

Left joints: [-0.43683845848166075, 1.4079920141688655, -0.34276521179916636, 0.9108873366158405, -0.1887399053106668, -0.5566727649235914, -1.0813536446581267]
Right joints: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [8]:
left_pose = server.left_robot_get_ee_pose()
right_pose = server.right_robot_get_ee_pose()
print("Left pose:", left_pose)
print("Right pose:", right_pose)

Left pose: [-0.42369599999999996, 0.25380199999999997, 0.015592, 1.6239765091031637, 0.1841845959629616, 0.03722787294503905]
Right pose: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [9]:
from nero_ik.ik_solver import fk
from pyAgxArm.utiles.tf import rot_to_rpy

In [10]:
ik_solver = server.left_ik_solver

T_fk = fk(left_joints, ik_solver.nero_params)
fk_xyz = np.asarray(T_fk[:3, 3], dtype=float)
fk_rpy = np.asarray(rot_to_rpy(T_fk[:3, :3].tolist()), dtype=float)
print("FK XYZ:", fk_xyz)
print("FK RPY:", fk_rpy)

FK XYZ: [-0.42369954  0.25380065  0.01559924]
FK RPY: [-1.62400178 -0.18420627 -3.10435915]


In [11]:
# left_arm_status = server.left_robot_get_arm_status()
# right_arm_status = server.right_robot_get_arm_status()

# print("Left arm status:", left_arm_status)
# print("Right arm status:", right_arm_status)

In [13]:
server.left_robot_go_home()

left_joints = server.left_robot_get_joint_positions()
print("Left joints:", left_joints)

Left joints: [0.0, -0.12997466939601773, 3.490658503988659e-05, 1.8699283072942046, 0.0, -3.490658503988659e-05, -0.16997761585172777]


In [ ]:
# left_joints = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints)

# left_joints_target = np.array(left_joints).copy()
# left_joints_target[0] -= np.deg2rad(0)
# left_joints_target[1] += np.deg2rad(0)
# left_joints_target[2] += np.deg2rad(0)
# left_joints_target[3] += np.deg2rad(0)
# left_joints_target[4] += np.deg2rad(0)
# left_joints_target[5] += np.deg2rad(0)
# left_joints_target[6] += np.deg2rad(0)

# print("Left joints target:", left_joints_target)

# server.left_robot_move_to_joint_positions(left_joints_target, delta=False)

# left_joints_real = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints_real)

In [ ]:
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

# pose_target = np.array(left_pose).copy()
# # end-effector pose [x, y, z, roll, pitch, yaw] (m, rad)
# # pose_target[0] += 0.01   # x方向移动1cm
# # pose_target[1] += 0.01
# # pose_target[2] += 0.01
# pose_target = [-0.4, 0.0, 0.4, 1.5708, 0.0, 0.0]

# server.left_robot_move_to_ee_pose(pose_target)
# time.sleep(3.0)
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

In [ ]:
# # 绝对控制
# target = [0, -70, 0, 100, 0, 0, 30]  # 7维绝对关节角度（度）

# ok = server.servo_j("left_robot", target, delta=False)
# print("servo_j ok:", ok)

In [ ]:
# 增量控制
target_delta = [10, 10, 10, 10, 10, 10, 10]

ok = server.servo_j("left_robot", target_delta, delta=True)
print("servo_j_delta ok:", ok)

In [ ]:
# # 单步 servo_p 测试

# # 构造一个小的 delta
# cur_pose = [-0.226,0,0.4,-1.57,0,-3.14]
# delta_pose = np.array([0.0, -0.00125, 0.0125, 0.0, 1.57, 0.0])
# # target_pose = np.array([-0.403, 0.03, 0.265, 1.57, -0.35, -0.07])

# # 调用 servo_p（delta 模式）
# ret = server.servo_p(
#     robot_arm="left_robot",
#     cur_pose=cur_pose,
#     pose=delta_pose.tolist(),
#     delta=True
#     # pose=target_pose.tolist(),
#     # delta=False
# )

In [ ]:
# # 连续 servo_p 测试
# steps = 1000
# dt = 0.02  # 20Hz
# # cur_pose = [-0.226,0,0.4,-1.57,0,-3.14]
# for i in range(steps):
#     # delta_pose = np.array([0.01, -0.00125, 0.0125, 0.0, 0.0, 0.0])
#     if(i < 500):
#         delta_pose = np.array([-0.0005, 0.0, 0.0, 0.0, 0.0, 0.0])
#     else:
#         delta_pose = np.array([0.0005, 0.0, 0.0, 0.0, 0.0, 0.0])
#     time1 = time.time()
#     server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
#     time2 = time.time()
#     print(f"Step {i+1}/{steps}, servo_p_OL time: {(time2 - time1) * 1000:.2f} ms")
#     time.sleep(dt)

In [ ]:
# # YZ 平面正方形轨迹测试
# # 正方形边长 0.2m，分为 20 段，每段 0.01m
# steps_per_side = 20  # 每边分20步
# side_length = 0.3   # 边长 0.3m
# step_size = side_length / steps_per_side  # 每步移动距离
# dt = 0.02  # 20Hz 控制周期

# # 当前起始位姿 (x, y, z, rx, ry, rz)
# # 保持 x 不变，在 YZ 平面运动
# cur_pose = [-0.226, 0.0, 0.4, -1.57, 0.0, -3.14]

# print(f"起始位姿: {cur_pose}")
# print(f"正方形边长: {side_length}m, 每边步数: {steps_per_side}, 每步距离: {step_size}m")

# # 正方形轨迹: 
# # 边1: y += 0.2 (向前)
# # 边2: z += 0.2 (向上)  
# # 边3: y -= 0.2 (向后)
# # 边4: z -= 0.2 (向下)

# # 边1: y 方向 +0.2m
# print(f"边1: y 方向 +0.2m")
# for i in range(steps_per_side):
#     cur_pose[1] = cur_pose[1]+step_size
#     delta_pose = np.array([0.0, step_size, 0.0, 0.0, 0.0, 0.0])
#     server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
#     time.sleep(dt)

# # 边2: z 方向 +0.2m
# print(f"边2: z 方向 +0.2m")
# for i in range(steps_per_side):
#     cur_pose[2] = cur_pose[2]+step_size
#     delta_pose = np.array([0.0, 0.0, step_size, 0.0, 0.0, 0.0])
#     server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
#     time.sleep(dt)

# # 边3: y 方向 -0.2m
# print(f"边3: y 方向 -0.2m")
# for i in range(steps_per_side):
#     cur_pose[1] = cur_pose[1]-step_size
#     delta_pose = np.array([0.0, -step_size, 0.0, 0.0, 0.0, 0.0])
#     server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
#     time.sleep(dt)

# # 边4: z 方向 -0.2m
# print(f"边4: z 方向 -0.2m")
# for i in range(steps_per_side):
#     cur_pose[2] = cur_pose[2]-step_size
#     delta_pose = np.array([0.0, 0.0, -step_size, 0.0, 0.0, 0.0])
#     server.servo_p_OL("left_robot", delta_pose.tolist(), delta=True)
#     time.sleep(dt)

# print("正方形轨迹完成!")

In [ ]:
# server.left_gripper_goto(0.05)
# # server.left_gripper_grasp()
# print(server.left_gripper_get_state())

In [ ]:
# server.stop("left_robot")